# Phase 10: Explainable AI (SHAP)

This notebook explains **why** the champion model (LightGBM) makes the predictions it makes — both overall (which features drive fraud detection in general) and for individual transactions (why *this specific* transaction was flagged or missed).

- **Global explainability**: SHAP summary plots across the full test set — which features matter most, and in which direction.
- **Local explainability**: SHAP waterfall plots for four specific transactions — a correctly caught fraud, a missed fraud, a false alarm, and a confidently-correct legit transaction — showing exactly which feature values pushed the score up or down.

This uses `shap.TreeExplainer`, which is exact (not approximated) for tree-based models like LightGBM and fast enough to run on the entire test set.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

shap.initjs()

## Phase 10, Step 1: Setup — Load Model, Test Data & Threshold

Loads the champion LightGBM model and the test set. Also loads `threshold_config.pkl` from Phase 9 if it exists, so the local examples below (TP/FN/FP/TN) are picked using your actual chosen decision threshold rather than the arbitrary 0.5 default. If you haven't run notebook `06_evaluation_and_threshold.ipynb` yet, this falls back to 0.5 with a warning — SHAP itself doesn't depend on the threshold, only the choice of *which* four example transactions get shown below does.

In [ ]:
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

best_supervised_model = joblib.load('../models/best_supervised_model.pkl')

try:
    threshold_config = joblib.load('../models/threshold_config.pkl')
    FINAL_THRESHOLD = threshold_config['final_threshold']
    print(f"Loaded Phase 9 threshold_config.pkl -> using decision threshold = {FINAL_THRESHOLD}")
except FileNotFoundError:
    FINAL_THRESHOLD = 0.5
    print("threshold_config.pkl not found (run notebook 06 first for the real threshold) "
          "-> using default 0.5 for now. SHAP values themselves are unaffected.")

print(f"\nModel: {type(best_supervised_model).__name__}")
print(f"Test set: {X_test.shape[0]} transactions, {int(y_test.sum())} fraud")

## Phase 10, Step 2: Build the SHAP Explainer & Compute Values

`TreeExplainer` reads the LightGBM tree structure directly, so it needs no background dataset and gives exact (not sampled/approximated) SHAP values. It's run once on the **entire test set** — this takes under a minute even for ~42K rows, since it only walks the trees.

Values are on the model's raw log-odds (margin) scale: `base_value + sum(shap_values for a row)` passed through a sigmoid reproduces `predict_proba` exactly — this was verified against the real model before writing this notebook.

In [ ]:
explainer = shap.TreeExplainer(best_supervised_model)

shap_values_test = explainer.shap_values(X_test)
base_value = explainer.expected_value

print(f"SHAP values shape: {shap_values_test.shape}")
print(f"Base value (log-odds of average prediction): {base_value:.4f}")

# Sanity check: SHAP values should reconstruct predict_proba via sigmoid
check_proba = 1 / (1 + np.exp(-(base_value + shap_values_test[:5].sum(axis=1))))
direct_proba = best_supervised_model.predict_proba(X_test.iloc[:5])[:, 1]
assert np.allclose(check_proba, direct_proba, atol=1e-4), 'SHAP reconstruction mismatch!'
print("Verified: base_value + SHAP values reconstruct predict_proba exactly.")

## Phase 10, Step 3: Global Feature Importance

Mean absolute SHAP value per feature across all test transactions — the single clearest ranking of "what matters most" to the model, independent of direction.

In [ ]:
mean_abs_shap = np.abs(shap_values_test).mean(axis=0)
importance_df = pd.DataFrame({
    'feature': X_test.columns,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(importance_df.head(15).to_string(index=False))

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values_test, X_test, plot_type='bar', show=False)
plt.title('Global Feature Importance — Mean |SHAP value|')
plt.tight_layout()
plt.show()

## Phase 10, Step 4: Beeswarm Plot — Direction, Not Just Magnitude

The bar chart above shows *how much* each feature matters but not *which way*. The beeswarm plot shows both: each dot is one transaction, its position shows whether that feature pushed the prediction toward fraud (right) or legit (left), and its color shows whether the feature's value was high (red) or low (blue) for that transaction.

Plotted on a random sample of 2,000 test transactions for readability (the ranking/importance numbers above already use the full test set).

In [ ]:
rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_test), size=min(2000, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]
shap_sample = shap_values_test[sample_idx]

plt.figure(figsize=(9, 6))
shap.summary_plot(shap_sample, X_sample, show=False)
plt.title('SHAP Beeswarm — Feature Value vs. Impact on Fraud Score (sample of 2,000)')
plt.tight_layout()
plt.show()

## Phase 10, Step 5: Dependence Plots — Top 4 Features

For each of the top 4 features, this shows how the feature's raw value relates to its SHAP contribution, and which other feature it interacts with most strongly (auto-selected, shown by color).

In [ ]:
top_features = importance_df['feature'].head(4).tolist()

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, feat in zip(axes, top_features):
    shap.dependence_plot(feat, shap_sample, X_sample, ax=ax, show=False)
    ax.set_title(feat)
plt.tight_layout()
plt.show()

print(f"Top 4 features by importance: {top_features}")

## Phase 10, Step 6: Local Explainability — Four Individual Transactions

Global plots explain the model in aggregate. For an actual fraud analyst, the useful question is: *why did the model flag (or miss) this one transaction?* Below are waterfall plots for four representative cases, picked using the Phase 9 decision threshold:

1. **Correctly caught fraud** (True Positive) — highest-confidence catch
2. **Missed fraud** (False Negative) — the closest call the model got wrong, most useful for understanding blind spots
3. **False alarm** (False Positive) — highest-confidence false alarm, most useful for understanding what triggers over-flagging
4. **Confidently correct legit** (True Negative) — a typical "obviously fine" transaction, as a baseline contrast

Each waterfall shows the base rate (average prediction) on the left, then each feature's contribution stacking up (red = pushes toward fraud, blue = pushes toward legit) to the final prediction on the right.

In [ ]:
y_proba_test = best_supervised_model.predict_proba(X_test)[:, 1]
y_pred_test = (y_proba_test >= FINAL_THRESHOLD).astype(int)
y_true = y_test.values

tp_mask = (y_pred_test == 1) & (y_true == 1)
fn_mask = (y_pred_test == 0) & (y_true == 1)
fp_mask = (y_pred_test == 1) & (y_true == 0)
tn_mask = (y_pred_test == 0) & (y_true == 0)

print(f"At threshold {FINAL_THRESHOLD}: TP={tp_mask.sum()}  FN={fn_mask.sum()}  "
      f"FP={fp_mask.sum()}  TN={tn_mask.sum()}")

def pick_example(mask, extreme='max'):
    idxs = np.where(mask)[0]
    if len(idxs) == 0:
        return None
    return idxs[np.argmax(y_proba_test[idxs])] if extreme == 'max' else idxs[np.argmin(y_proba_test[idxs])]

examples = {
    'Correctly caught fraud (TP)': pick_example(tp_mask, 'max'),
    'Missed fraud (FN)': pick_example(fn_mask, 'max'),
    'False alarm (FP)': pick_example(fp_mask, 'max'),
    'Confidently correct legit (TN)': pick_example(tn_mask, 'min')
}

for name, idx in examples.items():
    if idx is None:
        print(f"{name}: none found in test set at this threshold — skipping.")
    else:
        print(f"{name}: row {idx}, predicted probability = {y_proba_test[idx]:.6f}, actual class = {y_true[idx]}")

In [ ]:
for name, idx in examples.items():
    if idx is None:
        continue
    single_explanation = shap.Explanation(
        values=shap_values_test[idx],
        base_values=base_value,
        data=X_test.iloc[idx].values,
        feature_names=list(X_test.columns)
    )
    plt.figure(figsize=(9, 5.5))
    shap.plots.waterfall(single_explanation, show=False)
    plt.title(f"{name} — probability={y_proba_test[idx]:.4f}, actual={'Fraud' if y_true[idx]==1 else 'Legit'}")
    plt.tight_layout()
    plt.show()

## Phase 10, Step 7: Save Artifacts for Phase 11 Onward

Saves the global feature-importance ranking (used to justify feature choices in your report, and useful context for Phase 11 risk-score explanations).

The SHAP explainer itself is **not** pickled — for tree models it's near-instant to recreate (`shap.TreeExplainer(model)` takes ~0.1s, verified above), so Phase 12's FastAPI endpoint should just build it once at startup rather than loading a saved copy. This keeps the saved artifacts small and avoids any risk of an explainer pickled with a mismatched SHAP version breaking later.

In [ ]:
importance_df.to_csv('../models/shap_feature_importance.csv', index=False)
print("Saved models/shap_feature_importance.csv")

print("\nPhase 10 complete.")
print(f"Top 5 most important features: {importance_df['feature'].head(5).tolist()}")
print("Ready for Phase 11: Risk Scoring (0-30 Low, 30-70 Medium, 70-100 High).")